## Run this notebook with shortcake_default, after installing GraphST and POT 
```pip install POT GraphST```

In [ ]:
import glob
import os
import subprocess

import numpy as np
import scanpy as sc
import scipy.sparse as sp
import torch

from GraphST import GraphST
from GraphST.utils import clustering

## GraphST part

In [ ]:
input_dir   = "results/intermediate/CARD_anndata"
output_dir  = "results/intermediate/GraphST"
cluster_list = [12] # only this one is used further, initially wider search
device       = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
tool         = "leiden"   # "leiden", "louvain", or "mclust"
# radius       = 50         # only used if tool=="mclust"

r_home = subprocess.check_output(['R', 'RHOME']).decode().strip()
os.environ['R_HOME'] = r_home

os.makedirs(output_dir, exist_ok=True)

# find all .h5ad files
files = glob.glob(os.path.join(input_dir, "*step1.h5ad"))
print(f"Found {len(files)} files in {input_dir}")

In [ ]:
# Time-intensive GraphST domain searching, logging for errors kept

log_path = os.path.join(output_dir, "graphst_errors.log")
open(log_path, "w").close()

for fn in files:
    base = os.path.splitext(os.path.basename(fn))[0]
    print(f"\nProcessing {base}")

    # 1) Read & preprocess once
    adata0 = sc.read_h5ad(fn)
    coords = adata0.obsm['spatial']
    if hasattr(coords, "to_numpy"):
        coords = coords.to_numpy()
    adata0.obsm['spatial'] = np.asarray(coords, dtype=float)
    if sp.issparse(adata0.X):
        adata0.X = adata0.X.toarray().astype(float)
    else:
        adata0.X = np.asarray(adata0.X, dtype=float)
    sc.pp.highly_variable_genes(adata0, flavor="cell_ranger", n_top_genes=3000)

    # 2) Train once
    model = GraphST.GraphST(adata0, device=device)
    adata_tr = model.train()

    # 3) Loop cluster sizes
    for n_clusters in cluster_list:
        png_out = os.path.join(output_dir, f"{base}_k{n_clusters}.png")
        h5ad_out = os.path.join(output_dir, f"{base}_k{n_clusters}.h5ad")

        # skip if already done
        if os.path.exists(h5ad_out):
            print(f" → {base} k{n_clusters} already exists, skipping.")
            continue

        print(f"Clustering with n_clusters = {n_clusters}")
        ad = adata_tr.copy()

        try:
            if tool == 'mclust':
                clustering(ad, n_clusters, radius=radius,
                           method=tool, refinement=True)
            else:
                clustering(ad, n_clusters, radius=radius,
                           method=tool, start=0.1, end=2.0,
                           increment=0.01, refinement=False)

            # plot & save - commented out to save space
            # fig = sc.pl.spatial(ad, img_key="hires",
            #                     color=["domain"],
            #                     show=False, spot_size=5)
            # plt.savefig(png_out, bbox_inches="tight")
            # plt.close()

            # save AnnData
            ad.write_h5ad(h5ad_out, compression="gzip")

        except Exception as e:
            # log and continue
            msg = f"{base} k{n_clusters} ERROR: {e}\n"
            print("   !", msg.strip())
            with open(log_path, "a") as lf:
                lf.write(msg)
            continue
